# Rational Trade Check

This notebook checks whether accepted trades in one or more run-output folders were rational under the agents' latent utility functions.

For every accepted trade, it computes each involved player's utility before and after the trade:

\[
\Delta U_i = U_i(\text{after}) - U_i(\text{before})
\]

A trade is classified as:

- **Mutual gain**: both players increase utility.
- **One-sided loss**: one player gains, one player loses.
- **Mutual loss**: both players lose.
- **No change / weak change**: one or both players have near-zero utility change.

This is useful for detecting cases where an agent accepted a trade that lowered its own latent score.


In [2]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yaml
except ImportError as exc:
    raise ImportError("Please install PyYAML: pip install pyyaml") from exc


## 1. Configure run folders

Use either:

1. `RUN_DIRS`: manually list specific run output folders.
2. `RUN_PARENT`: point to a parent folder containing many run folders.

Each run folder is expected to contain:

```text
summary.json
trades.json
config_snapshot/players.yaml
```


In [5]:
# ---------------------------------------------------------------------
# User inputs
# ---------------------------------------------------------------------

RUN_DIRS = [
    #"/home/yyspencer/PSU/REAL/Preference_drift_results/Runs_6_3/silent barter runs",
    #"/home/yyspencer/PSU/REAL/Preference_drift_results/Runs_6_3/broadcast runs", 
]

# Optional: collect all immediate child run folders under this parent.
RUN_PARENT = "/home/yyspencer/PSU/REAL/Preference_drift_results/Runs_6_3/broadcast runs"

OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UTILITY_SHIFT = 1.0
TOL = 1e-9


## 2. Loading helpers

In [6]:
def collect_run_dirs(parent: Optional[str | Path]) -> List[Path]:
    if parent is None:
        return []
    parent = Path(parent)
    if not parent.exists():
        raise FileNotFoundError(parent)
    return sorted([
        p for p in parent.iterdir()
        if p.is_dir() and (p / "summary.json").exists()
    ])


def load_json(path: Path, default=None):
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_yaml(path: Path, default=None):
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def get_run_dirs() -> List[Path]:
    dirs = [Path(p) for p in RUN_DIRS]
    dirs.extend(collect_run_dirs(RUN_PARENT))

    # Keep only valid-looking run dirs.
    valid = []
    for p in dirs:
        if (p / "summary.json").exists() and (p / "trades.json").exists():
            valid.append(p)
        else:
            print(f"[SKIP] Missing summary.json or trades.json: {p}")

    # Remove duplicates while preserving order.
    seen = set()
    unique = []
    for p in valid:
        resolved = str(p.resolve())
        if resolved not in seen:
            seen.add(resolved)
            unique.append(p)
    return unique


def run_id(run_dir: Path) -> str:
    return Path(run_dir).name


def load_summary(run_dir: Path) -> Dict[str, Any]:
    return load_json(run_dir / "summary.json")


def load_trades(run_dir: Path) -> List[Dict[str, Any]]:
    return load_json(run_dir / "trades.json", default=[])


def load_players_yaml(run_dir: Path) -> Dict[str, Any]:
    return load_yaml(run_dir / "config_snapshot" / "players.yaml")


run_dirs = get_run_dirs()
print(f"Loaded {len(run_dirs)} run folders.")
for p in run_dirs:
    print(" -", p)


Loaded 8 run folders.
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_171055_preference_drift_v1_parallel_pairs_run_1
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_171326_preference_drift_v1_parallel_pairs_run_2
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_171550_preference_drift_v1_parallel_pairs_run_3
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_171851_preference_drift_v1_parallel_pairs_run_4
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_172157_preference_drift_v1_parallel_pairs_run_5
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_172440_preference_drift_v1_parallel_pairs_run_6
 - \home\yyspencer\PSU\REAL\Preference_drift_results\Runs_6_3\broadcast runs\20260603_172744_preference_drift_v1_parallel_pairs_run_7
 - \home\yyspencer\PSU\REAL\Preference_d

## 3. Utility and player helpers

The notebook recomputes utility from `players.yaml` and the inventories saved in `trades.json`.

By default it uses the same shifted Cobb-Douglas utility used in the experiment:

\[
U_i(x) = \prod_g (x_g + 1)^{\alpha_{ig}}
\]


In [7]:
def shifted_cobb_douglas(
    inventory: Mapping[str, int],
    weights: Mapping[str, float],
    shift: float = 1.0,
) -> float:
    utility = 1.0
    for good, alpha in weights.items():
        utility *= (inventory.get(good, 0) + shift) ** alpha
    return float(utility)


def load_player_specs(run_dir: Path) -> Dict[str, Dict[str, Any]]:
    players_yaml = load_players_yaml(run_dir)
    players = players_yaml.get("players", [])
    return {p["id"]: p for p in players}


def get_player_weights(player_specs: Dict[str, Dict[str, Any]], player_id: str) -> Dict[str, float]:
    if player_id not in player_specs:
        raise KeyError(f"Player {player_id} not found in players.yaml.")
    return dict(player_specs[player_id]["utility_weights"])


def get_player_role(player_specs: Dict[str, Dict[str, Any]], player_id: str) -> str:
    return player_specs.get(player_id, {}).get("role", "")


def get_player_name(player_specs: Dict[str, Dict[str, Any]], player_id: str) -> str:
    return player_specs.get(player_id, {}).get("display_name", player_id)


def utility_delta_from_trade_record(
    trade: Mapping[str, Any],
    player_id: str,
    player_specs: Dict[str, Dict[str, Any]],
    shift: float = 1.0,
) -> Dict[str, Any]:
    before_inv = trade["inventory_before"][player_id]
    after_inv = trade["inventory_after"][player_id]
    weights = get_player_weights(player_specs, player_id)

    u_before = shifted_cobb_douglas(before_inv, weights, shift=shift)
    u_after = shifted_cobb_douglas(after_inv, weights, shift=shift)

    logged_before = None
    logged_after = None

    if "utility_before" in trade and player_id in trade["utility_before"]:
        logged_before = trade["utility_before"][player_id]
    if "utility_after" in trade and player_id in trade["utility_after"]:
        logged_after = trade["utility_after"][player_id]

    return {
        "player_id": player_id,
        "display_name": get_player_name(player_specs, player_id),
        "role": get_player_role(player_specs, player_id),
        "inventory_before": before_inv,
        "inventory_after": after_inv,
        "utility_before_recomputed": u_before,
        "utility_after_recomputed": u_after,
        "utility_delta_recomputed": u_after - u_before,
        "logged_utility_before": logged_before,
        "logged_utility_after": logged_after,
        "logged_utility_delta": (
            logged_after - logged_before
            if logged_before is not None and logged_after is not None
            else np.nan
        ),
        "utility_recompute_abs_error_before": (
            abs(u_before - logged_before) if logged_before is not None else np.nan
        ),
        "utility_recompute_abs_error_after": (
            abs(u_after - logged_after) if logged_after is not None else np.nan
        ),
    }


## 4. Extract trade rationality rows

Each accepted trade becomes two rows: one for the proposer and one for the responder.

The notebook also labels the trade-level outcome:

- `mutual_gain`
- `one_sided_loss`
- `mutual_loss`
- `weak_or_no_change`


In [8]:
def classify_trade_player_delta(delta: float, tol: float = 1e-9) -> str:
    if delta > tol:
        return "gain"
    if delta < -tol:
        return "loss"
    return "no_change"


def classify_trade_outcome(deltas: List[float], tol: float = 1e-9) -> str:
    statuses = [classify_trade_player_delta(d, tol=tol) for d in deltas]

    if all(s == "gain" for s in statuses):
        return "mutual_gain"
    if all(s == "loss" for s in statuses):
        return "mutual_loss"
    if "loss" in statuses and "gain" in statuses:
        return "one_sided_loss"
    if "loss" in statuses:
        return "loss_with_no_change"
    return "weak_or_no_change"


def infer_trade_players(trade: Mapping[str, Any]) -> List[str]:
    if "inventory_before" in trade and isinstance(trade["inventory_before"], dict):
        return list(trade["inventory_before"].keys())

    players = []
    for key in ["proposer_id", "responder_id"]:
        if key in trade and trade[key] is not None:
            players.append(trade[key])
    return players


def extract_rationality_rows_for_run(run_dir: Path, shift: float = 1.0, tol: float = 1e-9) -> pd.DataFrame:
    summary = load_summary(run_dir)
    trades = load_trades(run_dir)
    player_specs = load_player_specs(run_dir)

    condition = summary.get("experiment_name", "")
    mode = summary.get("mode", "")

    rows = []

    for trade_index, trade in enumerate(trades):
        accepted = bool(trade.get("accepted", False))
        if not accepted:
            continue

        trade_players = infer_trade_players(trade)
        if len(trade_players) != 2:
            print(f"[WARN] Accepted trade without exactly two players in {run_id(run_dir)} trade index {trade_index}")
            continue

        player_delta_records = []
        for pid in trade_players:
            rec = utility_delta_from_trade_record(
                trade=trade,
                player_id=pid,
                player_specs=player_specs,
                shift=shift,
            )
            player_delta_records.append(rec)

        deltas = [rec["utility_delta_recomputed"] for rec in player_delta_records]
        trade_outcome = classify_trade_outcome(deltas, tol=tol)

        for rec in player_delta_records:
            delta = rec["utility_delta_recomputed"]
            rows.append({
                "run_id": run_id(run_dir),
                "run_dir": str(run_dir),
                "condition": condition,
                "mode": mode,
                "round_index": trade.get("round_index"),
                "pair_id": trade.get("pair_id"),
                "trade_index": trade_index,
                "accepted": accepted,
                "trade_outcome": trade_outcome,
                "player_trade_outcome": classify_trade_player_delta(delta, tol=tol),
                "player_id": rec["player_id"],
                "display_name": rec["display_name"],
                "role": rec["role"],
                "proposer_id": trade.get("proposer_id"),
                "responder_id": trade.get("responder_id"),
                "is_proposer": rec["player_id"] == trade.get("proposer_id"),
                "is_responder": rec["player_id"] == trade.get("responder_id"),
                "proposed_trade": json.dumps(trade.get("proposed_trade", {}), ensure_ascii=False),
                "inventory_before": json.dumps(rec["inventory_before"], ensure_ascii=False),
                "inventory_after": json.dumps(rec["inventory_after"], ensure_ascii=False),
                "utility_before": rec["utility_before_recomputed"],
                "utility_after": rec["utility_after_recomputed"],
                "utility_delta": delta,
                "logged_utility_before": rec["logged_utility_before"],
                "logged_utility_after": rec["logged_utility_after"],
                "logged_utility_delta": rec["logged_utility_delta"],
                "utility_recompute_abs_error_before": rec["utility_recompute_abs_error_before"],
                "utility_recompute_abs_error_after": rec["utility_recompute_abs_error_after"],
            })

    return pd.DataFrame(rows)


dfs = []
for rd in run_dirs:
    dfs.append(extract_rationality_rows_for_run(rd, shift=UTILITY_SHIFT, tol=TOL))

rationality_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if not rationality_df.empty:
    rationality_df.to_csv(OUTPUT_DIR / "accepted_trade_rationality_player_level.csv", index=False)

rationality_df.head()


,run_id,run_dir,condition,mode,round_index,pair_id,trade_index,accepted,trade_outcome,player_trade_outcome,...,inventory_before,inventory_after,utility_before,utility_after,utility_delta,logged_utility_before,logged_utility_after,logged_utility_delta,utility_recompute_abs_error_before,utility_recompute_abs_error_after
0,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,1,round_1_player_4_vs_player_6,0,True,mutual_gain,gain,...,"{""A"": 3, ""B"": 1, ""C"": 0}","{""A"": 2, ""B"": 2, ""C"": 0}",1.802501,2.279507,0.477006,1.802501,2.279507,0.477006,0.0,0.0
1,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,1,round_1_player_4_vs_player_6,0,True,mutual_gain,gain,...,"{""A"": 0, ""B"": 2, ""C"": 2}","{""A"": 1, ""B"": 1, ""C"": 2}",2.064905,2.270543,0.205639,2.064905,2.270543,0.205639,0.0,0.0
2,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,1,round_1_player_3_vs_player_5,2,True,mutual_gain,gain,...,"{""A"": 3, ""B"": 1, ""C"": 0}","{""A"": 2, ""B"": 2, ""C"": 0}",1.802501,2.279507,0.477006,1.802501,2.279507,0.477006,0.0,0.0
3,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,1,round_1_player_3_vs_player_5,2,True,mutual_gain,gain,...,"{""A"": 0, ""B"": 2, ""C"": 2}","{""A"": 1, ""B"": 1, ""C"": 2}",2.064905,2.270543,0.205639,2.064905,2.270543,0.205639,0.0,0.0
4,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,2,round_2_player_4_vs_player_1,3,True,mutual_gain,gain,...,"{""A"": 2, ""B"": 2, ""C"": 0}","{""A"": 1, ""B"": 2, ""C"": 1}",2.279507,2.603091,0.323584,2.279507,2.603091,0.323584,0.0,0.0


## 5. Run-level and trade-level summaries

In [9]:
def summarize_rationality(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # Player-level summary: each accepted trade contributes two rows.
    player_summary = (
        df
        .groupby(["run_id", "condition", "mode"], as_index=False)
        .agg(
            accepted_trade_player_decisions=("player_id", "count"),
            player_gains=("player_trade_outcome", lambda s: (s == "gain").sum()),
            player_losses=("player_trade_outcome", lambda s: (s == "loss").sum()),
            player_no_changes=("player_trade_outcome", lambda s: (s == "no_change").sum()),
            mean_player_delta=("utility_delta", "mean"),
            min_player_delta=("utility_delta", "min"),
            max_player_delta=("utility_delta", "max"),
        )
    )
    player_summary["loss_rate_player_level"] = (
        player_summary["player_losses"] / player_summary["accepted_trade_player_decisions"]
    )

    # Trade-level summary: collapse two rows per trade into one.
    trade_level = (
        df
        .groupby(["run_id", "condition", "mode", "round_index", "pair_id", "trade_index"], as_index=False)
        .agg(
            trade_outcome=("trade_outcome", "first"),
            min_delta=("utility_delta", "min"),
            max_delta=("utility_delta", "max"),
            sum_delta=("utility_delta", "sum"),
            proposed_trade=("proposed_trade", "first"),
            proposer_id=("proposer_id", "first"),
            responder_id=("responder_id", "first"),
        )
    )

    trade_summary = (
        trade_level
        .groupby(["run_id", "condition", "mode"], as_index=False)
        .agg(
            accepted_trades=("pair_id", "count"),
            mutual_gain_trades=("trade_outcome", lambda s: (s == "mutual_gain").sum()),
            one_sided_loss_trades=("trade_outcome", lambda s: (s == "one_sided_loss").sum()),
            mutual_loss_trades=("trade_outcome", lambda s: (s == "mutual_loss").sum()),
            weak_or_no_change_trades=("trade_outcome", lambda s: (s == "weak_or_no_change").sum()),
            mean_trade_sum_delta=("sum_delta", "mean"),
            min_trade_min_delta=("min_delta", "min"),
        )
    )

    trade_summary["irrational_trade_rate"] = (
        (trade_summary["one_sided_loss_trades"] + trade_summary["mutual_loss_trades"])
        / trade_summary["accepted_trades"]
    )

    # Role-level summary.
    role_summary = (
        df
        .groupby(["condition", "role"], as_index=False)
        .agg(
            n_player_trade_decisions=("player_id", "count"),
            losses=("player_trade_outcome", lambda s: (s == "loss").sum()),
            gains=("player_trade_outcome", lambda s: (s == "gain").sum()),
            mean_delta=("utility_delta", "mean"),
            min_delta=("utility_delta", "min"),
        )
    )
    role_summary["loss_rate"] = role_summary["losses"] / role_summary["n_player_trade_decisions"]

    return player_summary, trade_summary, role_summary


player_summary, trade_summary, role_summary = summarize_rationality(rationality_df)

if not player_summary.empty:
    player_summary.to_csv(OUTPUT_DIR / "accepted_trade_rationality_run_player_summary.csv", index=False)
if not trade_summary.empty:
    trade_summary.to_csv(OUTPUT_DIR / "accepted_trade_rationality_run_trade_summary.csv", index=False)
if not role_summary.empty:
    role_summary.to_csv(OUTPUT_DIR / "accepted_trade_rationality_role_summary.csv", index=False)

display(player_summary)
display(trade_summary)
display(role_summary)


,run_id,condition,mode,accepted_trade_player_decisions,player_gains,player_losses,player_no_changes,mean_player_delta,min_player_delta,max_player_delta,loss_rate_player_level
0,20260603_171055_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_1,gpt,20,13,4,3,0.182813,-0.284358,0.542159,0.200000
1,20260603_171326_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_2,gpt,20,14,2,4,0.193915,-0.284358,0.573643,0.100000
2,20260603_171550_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_3,gpt,20,12,4,4,0.176848,-0.285358,0.542159,0.200000
3,20260603_171851_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_4,gpt,18,13,3,2,0.192613,-0.205639,0.542159,0.166667
4,20260603_172157_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_5,gpt,18,13,3,2,0.215194,-0.205639,0.542159,0.166667
5,20260603_172440_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_6,gpt,18,15,3,0,0.202652,-0.284358,0.573643,0.166667
6,20260603_172744_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_7,gpt,20,15,4,1,0.199067,-0.284358,0.542159,0.200000
7,20260603_173026_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_8,gpt,20,15,2,3,0.205337,-0.284358,0.573643,0.100000


,run_id,condition,mode,accepted_trades,mutual_gain_trades,one_sided_loss_trades,mutual_loss_trades,weak_or_no_change_trades,mean_trade_sum_delta,min_trade_min_delta,irrational_trade_rate
0,20260603_171055_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_1,gpt,10,5,1,1,2,0.365625,-0.284358,0.200000
1,20260603_171326_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_2,gpt,10,4,2,0,4,0.387831,-0.284358,0.200000
2,20260603_171550_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_3,gpt,10,4,1,1,3,0.353696,-0.285358,0.200000
3,20260603_171851_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_4,gpt,9,5,1,1,2,0.385226,-0.205639,0.222222
4,20260603_172157_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_5,gpt,9,5,1,1,2,0.430388,-0.205639,0.222222
5,20260603_172440_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_6,gpt,9,6,3,0,0,0.405304,-0.284358,0.333333
6,20260603_172744_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_7,gpt,10,5,4,0,1,0.398133,-0.284358,0.400000
7,20260603_173026_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_8,gpt,10,5,2,0,3,0.410675,-0.284358,0.200000


,condition,role,n_player_trade_decisions,losses,gains,mean_delta,min_delta,loss_rate
0,preference_drift_v1_parallel_pairs_run_1,Builder,5,0,5,0.425546,0.203006,0.000000
1,preference_drift_v1_parallel_pairs_run_1,Merchant,7,2,2,-0.022491,-0.284358,0.285714
2,preference_drift_v1_parallel_pairs_run_1,Weaver,8,2,6,0.210746,-0.240296,0.250000
3,preference_drift_v1_parallel_pairs_run_2,Builder,5,0,5,0.384944,0.172025,0.000000
4,preference_drift_v1_parallel_pairs_run_2,Merchant,9,2,3,0.045697,-0.284358,0.222222
5,preference_drift_v1_parallel_pairs_run_2,Weaver,6,0,6,0.257051,0.005775,0.000000
6,preference_drift_v1_parallel_pairs_run_3,Builder,5,0,5,0.401888,0.248178,0.000000
7,preference_drift_v1_parallel_pairs_run_3,Merchant,7,2,1,-0.022634,-0.285358,0.285714
8,preference_drift_v1_parallel_pairs_run_3,Weaver,8,2,6,0.210746,-0.240296,0.250000
9,preference_drift_v1_parallel_pairs_run_4,Builder,6,1,5,0.320787,-0.172025,0.166667


## 6. Flag suspicious accepted trades

These are accepted trades where at least one participant lost latent utility.


In [10]:
if rationality_df.empty:
    suspicious_trades = pd.DataFrame()
else:
    trade_level = (
        rationality_df
        .groupby(["run_id", "condition", "round_index", "pair_id", "trade_index"], as_index=False)
        .agg(
            trade_outcome=("trade_outcome", "first"),
            min_delta=("utility_delta", "min"),
            max_delta=("utility_delta", "max"),
            sum_delta=("utility_delta", "sum"),
            proposed_trade=("proposed_trade", "first"),
            proposer_id=("proposer_id", "first"),
            responder_id=("responder_id", "first"),
        )
    )

    suspicious_trades = trade_level[
        trade_level["trade_outcome"].isin(["one_sided_loss", "mutual_loss", "loss_with_no_change"])
    ].copy()

    suspicious_trades = suspicious_trades.sort_values(
        ["condition", "run_id", "round_index", "pair_id"]
    )

    suspicious_trades.to_csv(
        OUTPUT_DIR / "suspicious_accepted_trades_trade_level.csv",
        index=False,
    )

suspicious_trades


,run_id,condition,round_index,pair_id,trade_index,trade_outcome,min_delta,max_delta,sum_delta,proposed_trade,proposer_id,responder_id
4,20260603_171055_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_1,2,round_2_player_6_vs_player_5,4,loss_with_no_change,-0.284358,-4.440892e-16,-0.284358,"{""give"": {""C"": 1}, ""receive"": {""A"": 1}}",player_6,player_5
8,20260603_171055_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_1,5,round_5_player_3_vs_player_6,13,mutual_loss,-0.284358,-8.478460e-02,-0.369143,"{""give"": {""C"": 1}, ""receive"": {""A"": 1}}",player_3,player_6
9,20260603_171055_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_1,5,round_5_player_4_vs_player_2,12,one_sided_loss,-0.240296,2.030060e-01,-0.037290,"{""give"": {""B"": 1}, ""receive"": {""C"": 1}}",player_4,player_2
12,20260603_171326_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_2,1,round_1_player_6_vs_player_1,0,one_sided_loss,-0.078720,2.166935e-01,0.137974,"{""give"": {""C"": 1}, ""receive"": {""B"": 1}}",player_1,player_6
16,20260603_171326_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_2,4,round_4_player_2_vs_player_5,10,one_sided_loss,-0.284358,4.202024e-01,0.135844,"{""give"": {""C"": 1}, ""receive"": {""A"": 1}}",player_2,player_5
24,20260603_171550_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_3,3,round_3_player_5_vs_player_6,7,loss_with_no_change,-0.078720,-4.440892e-16,-0.078720,"{""give"": {""C"": 1}, ""receive"": {""B"": 1}}",player_5,player_6
28,20260603_171550_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_3,5,round_5_player_3_vs_player_2,12,one_sided_loss,-0.240296,2.481778e-01,0.007882,"{""give"": {""B"": 1}, ""receive"": {""C"": 1}}",player_3,player_2
29,20260603_171550_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_3,5,round_5_player_6_vs_player_4,13,mutual_loss,-0.285358,-8.478460e-02,-0.370143,"{""give"": {""A"": 1}, ""receive"": {""C"": 1}}",player_6,player_4
36,20260603_171851_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_4,4,round_4_player_5_vs_player_1,10,mutual_loss,-0.205639,-1.720246e-01,-0.377663,"{""give"": {""B"": 1}, ""receive"": {""A"": 1}}",player_5,player_1
38,20260603_171851_preference_drift_v1_parallel_p...,preference_drift_v1_parallel_pairs_run_4,5,round_5_player_6_vs_player_1,12,one_sided_loss,-0.205639,1.720246e-01,-0.033614,"{""give"": {""B"": 1}, ""receive"": {""A"": 1}}",player_1,player_6


## 7. Player-level details for suspicious trades

This table shows which player lost utility in each suspicious trade.


In [11]:
if rationality_df.empty or suspicious_trades.empty:
    suspicious_player_details = pd.DataFrame()
else:
    suspicious_keys = suspicious_trades[
        ["run_id", "round_index", "pair_id", "trade_index"]
    ].drop_duplicates()

    suspicious_player_details = rationality_df.merge(
        suspicious_keys,
        on=["run_id", "round_index", "pair_id", "trade_index"],
        how="inner",
    ).sort_values(["condition", "run_id", "round_index", "pair_id", "utility_delta"])

    suspicious_player_details.to_csv(
        OUTPUT_DIR / "suspicious_accepted_trades_player_details.csv",
        index=False,
    )

suspicious_player_details


,run_id,run_dir,condition,mode,round_index,pair_id,trade_index,accepted,trade_outcome,player_trade_outcome,...,inventory_before,inventory_after,utility_before,utility_after,utility_delta,logged_utility_before,logged_utility_after,logged_utility_delta,utility_recompute_abs_error_before,utility_recompute_abs_error_after
1,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,2,round_2_player_6_vs_player_5,4,True,loss_with_no_change,loss,...,"{""A"": 1, ""B"": 1, ""C"": 2}","{""A"": 0, ""B"": 1, ""C"": 3}",2.270543,1.986185,-2.843584e-01,2.270543,1.986185,-2.843584e-01,0.0,0.0
0,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,2,round_2_player_6_vs_player_5,4,True,loss_with_no_change,no_change,...,"{""A"": 1, ""B"": 1, ""C"": 2}","{""A"": 2, ""B"": 1, ""C"": 1}",2.270543,2.270543,-4.440892e-16,2.270543,2.270543,-4.440892e-16,0.0,0.0
5,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,5,round_5_player_3_vs_player_6,13,True,mutual_loss,loss,...,"{""A"": 1, ""B"": 1, ""C"": 2}","{""A"": 0, ""B"": 1, ""C"": 3}",2.270543,1.986185,-2.843584e-01,2.270543,1.986185,-2.843584e-01,0.0,0.0
4,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,5,round_5_player_3_vs_player_6,13,True,mutual_loss,loss,...,"{""A"": 0, ""B"": 2, ""C"": 2}","{""A"": 1, ""B"": 2, ""C"": 1}",2.687875,2.603091,-8.478460e-02,2.687875,2.603091,-8.478460e-02,0.0,0.0
2,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,5,round_5_player_4_vs_player_2,12,True,one_sided_loss,loss,...,"{""A"": 0, ""B"": 3, ""C"": 1}","{""A"": 0, ""B"": 2, ""C"": 2}",2.928171,2.687875,-2.402960e-01,2.928171,2.687875,-2.402960e-01,0.0,0.0
3,20260603_171055_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_1,gpt,5,round_5_player_4_vs_player_2,12,True,one_sided_loss,gain,...,"{""A"": 3, ""B"": 0, ""C"": 1}","{""A"": 3, ""B"": 1, ""C"": 0}",2.828427,3.031433,2.030060e-01,2.828427,3.031433,2.030060e-01,0.0,0.0
6,20260603_171326_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_2,gpt,1,round_1_player_6_vs_player_1,0,True,one_sided_loss,loss,...,"{""A"": 0, ""B"": 2, ""C"": 2}","{""A"": 0, ""B"": 1, ""C"": 3}",2.064905,1.986185,-7.871978e-02,2.064905,1.986185,-7.871978e-02,0.0,0.0
7,20260603_171326_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_2,gpt,1,round_1_player_6_vs_player_1,0,True,one_sided_loss,gain,...,"{""A"": 1, ""B"": 0, ""C"": 3}","{""A"": 1, ""B"": 1, ""C"": 2}",1.866066,2.082759,2.166935e-01,1.866066,2.082759,2.166935e-01,0.0,0.0
9,20260603_171326_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_2,gpt,4,round_4_player_2_vs_player_5,10,True,one_sided_loss,loss,...,"{""A"": 1, ""B"": 1, ""C"": 2}","{""A"": 0, ""B"": 1, ""C"": 3}",2.270543,1.986185,-2.843584e-01,2.270543,1.986185,-2.843584e-01,0.0,0.0
8,20260603_171326_preference_drift_v1_parallel_p...,\home\yyspencer\PSU\REAL\Preference_drift_resu...,preference_drift_v1_parallel_pairs_run_2,gpt,4,round_4_player_2_vs_player_5,10,True,one_sided_loss,gain,...,"{""A"": 2, ""B"": 0, ""C"": 2}","{""A"": 3, ""B"": 0, ""C"": 1}",2.408225,2.828427,4.202024e-01,2.408225,2.828427,4.202024e-01,0.0,0.0


## 8. Visualizations

In [ ]:
def barplot_trade_outcomes(trade_summary: pd.DataFrame, out_path: Path):
    if trade_summary.empty:
        print("No trade summary data.")
        return

    condition_summary = (
        trade_summary
        .groupby("condition", as_index=False)
        .agg(
            accepted_trades=("accepted_trades", "sum"),
            mutual_gain_trades=("mutual_gain_trades", "sum"),
            one_sided_loss_trades=("one_sided_loss_trades", "sum"),
            mutual_loss_trades=("mutual_loss_trades", "sum"),
            weak_or_no_change_trades=("weak_or_no_change_trades", "sum"),
        )
    )

    condition_summary["irrational_or_loss_trades"] = (
        condition_summary["one_sided_loss_trades"] + condition_summary["mutual_loss_trades"]
    )
    condition_summary["irrational_trade_rate"] = (
        condition_summary["irrational_or_loss_trades"] / condition_summary["accepted_trades"]
    )

    x = np.arange(len(condition_summary))

    plt.figure(figsize=(8, 5))
    plt.bar(x, condition_summary["irrational_trade_rate"])
    plt.xticks(x, condition_summary["condition"], rotation=15, ha="right")
    plt.ylabel("Fraction of accepted trades with utility loss")
    plt.title("Accepted trades that harmed at least one participant")
    plt.ylim(0, max(0.05, condition_summary["irrational_trade_rate"].max() * 1.2))
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()

    return condition_summary


condition_trade_summary = barplot_trade_outcomes(
    trade_summary,
    OUTPUT_DIR / "accepted_trade_loss_rate_by_condition.png",
)
condition_trade_summary


In [ ]:
def plot_utility_delta_distribution(df: pd.DataFrame, out_path: Path):
    if df.empty:
        print("No rationality data.")
        return

    plt.figure(figsize=(8, 5))

    conditions = list(df["condition"].dropna().unique())
    data = [df[df["condition"] == c]["utility_delta"].dropna().to_numpy() for c in conditions]

    plt.boxplot(data, labels=conditions)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.ylabel("Player utility delta from accepted trade")
    plt.title("Distribution of player-level utility changes from accepted trades")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()


plot_utility_delta_distribution(
    rationality_df,
    OUTPUT_DIR / "accepted_trade_player_delta_distribution.png",
)


In [ ]:
def plot_role_loss_rates(role_summary: pd.DataFrame, out_path: Path):
    if role_summary.empty:
        print("No role summary data.")
        return

    pivot = role_summary.pivot(index="role", columns="condition", values="loss_rate").fillna(0)

    ax = pivot.plot(kind="bar", figsize=(8, 5))
    ax.set_ylabel("Loss rate among accepted trade decisions")
    ax.set_title("Which roles accept utility-losing trades?")
    ax.set_ylim(0, max(0.05, float(pivot.max().max()) * 1.2))
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()


plot_role_loss_rates(
    role_summary,
    OUTPUT_DIR / "accepted_trade_loss_rate_by_role.png",
)


## 9. Interpretation helper

Use these checks:

- If `irrational_trade_rate` is near zero, accepted trades are mostly individually rational.
- If one condition has a much higher `irrational_trade_rate`, that mechanism may be inducing bad trades.
- If Merchant loss rate is high, this supports the hypothesis that Merchants are being exploited or anchored by market information.
- If many losses are extremely small, inspect `TOL`; some are numerical noise.


In [ ]:
print("Saved outputs in:", OUTPUT_DIR.resolve())

if not rationality_df.empty:
    print("\nOverall player-level outcomes:")
    print(rationality_df["player_trade_outcome"].value_counts(dropna=False))

if not suspicious_trades.empty:
    print(f"\nSuspicious accepted trades: {len(suspicious_trades)}")
    display(suspicious_trades.head(20))
else:
    print("\nNo accepted trades with utility losses found under the current tolerance.")
